# Backend Choice Guide

NRPy separates symbolic equation construction from generated-project backends. Choose a backend from the target runtime, required ecosystem, and validation budget before writing project-specific code.

## Table of Contents

1. [Decision table](#Decision-table)
2. [Backend and example discovery](#Backend-and-example-discovery)
3. [Selection notes](#Selection-notes)

## Decision Table

| Backend | Use when | Validation mode | Dependency tier |
|---|---|---|---|
| BHaH | You need a standalone generated C project and dependency-light mainline examples. | Generate, build, run, and inspect project output. | Core Python plus a C compiler and `make`. |
| ETLegacy | You need Einstein Toolkit thorn generation in the older Cactus/Carpet style. | Generate thorns and inspect CCL/source files; full builds require an Einstein Toolkit checkout. | External Einstein Toolkit stack. |
| CarpetX | You need newer Einstein Toolkit thorn generation for CarpetX-oriented applications. | Generate thorns and inspect metadata/source files; full builds require the target Einstein Toolkit environment. | External Einstein Toolkit and CarpetX stack. |
| superB | You are targeting Charm++-oriented workflows. | Project-specific generation and external runtime validation. | Charm++ and application-specific dependencies. |
| JAX | You need a JAX-targeted generated project. | Import/generate in the JAX workflow, then validate with the target JAX runtime. | JAX-capable Python environment. |

## Backend and Example Discovery

This cell imports NRPy, records the package location for context, imports core backend packages that are part of the standard infrastructure path, and discovers representative example entry points without importing or running them.

In [ ]:
import importlib
import importlib.util

import nrpy

print("nrpy package:", nrpy.__file__)

backend_modules = {
    "BHaH": "nrpy.infrastructures.BHaH",
    "ETLegacy": "nrpy.infrastructures.ETLegacy",
    "CarpetX": "nrpy.infrastructures.CarpetX",
    "superB": "nrpy.infrastructures.superB",
    "JAX": "nrpy.infrastructures.JAX",
}

for backend, module_name in backend_modules.items():
    spec = importlib.util.find_spec(module_name)
    print(f"{backend} backend found:", spec is not None)
    if spec is not None and backend in {"BHaH", "ETLegacy", "CarpetX"}:
        module = importlib.import_module(module_name)
        print(f"{backend} package:", getattr(module, "__file__", "<namespace>"))

example_modules = [
    "nrpy.examples.wave_equation_cartesian",
    "nrpy.examples.wave_equation_curvilinear",
    "nrpy.examples.carpet_wavetoy_thorns",
    "nrpy.examples.carpetx_wavetoy_thorns",
    "nrpy.examples.two_blackholes_collide",
]

for module_name in example_modules:
    print(f"{module_name} found:", importlib.util.find_spec(module_name) is not None)

## Selection Notes

BHaH is the default choice for portable generated C projects and for examples that should build outside a large application framework. ETLegacy and CarpetX are the right choices when the deliverable is an Einstein Toolkit thorn. superB and JAX target specialized runtimes; choose them only when that runtime is part of the scientific or deployment requirement.